In [1]:
import zipfile
import pandas as pd
from io import BytesIO
from pathlib import Path

# === 输入输出路径 ===
zip_path = Path("/Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250928-df0cec/knowledge_graph.tsv.zip")
out_zip_path = zip_path.with_name("knowledge_graph_noascii.tsv.zip")

print(f"📦 正在处理：{zip_path.name}")

with zipfile.ZipFile(zip_path, "r") as zin, zipfile.ZipFile(out_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zout:
    for name in zin.namelist():
        if not name.lower().endswith(".tsv"):
            print(f"  跳过非TSV文件：{name}")
            zout.writestr(name, zin.read(name))
            continue

        print(f"  修改 TSV 文件：{name}")
        raw = zin.read(name)
        try:
            text = raw.decode("utf-8")
        except UnicodeDecodeError:
            text = raw.decode("latin-1", errors="replace")

        # 读取为 DataFrame
        df = pd.read_csv(BytesIO(raw), sep="\t", dtype=str, low_memory=False, encoding="utf-8")

        # 检查是否存在 pmids / key_sentence 列
        for col in ["pmids", "key_sentence"]:
            if col in df.columns:
                df[col] = ""

        # 写回新的 zip
        buffer = BytesIO()
        df.to_csv(buffer, sep="\t", index=False, encoding="utf-8")
        zout.writestr(name, buffer.getvalue())

print(f"✅ 已生成新文件：{out_zip_path}")


📦 正在处理：knowledge_graph.tsv.zip
  修改 TSV 文件：knowledge_graph.tsv
✅ 已生成新文件：/Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250928-df0cec/knowledge_graph_noascii.tsv.zip


In [4]:
import zipfile
from pathlib import Path

# === 你的 ZIP 文件路径 ===
zip_path = Path("/Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250831-8cdd6f/knowledge_graph_noascii.zip")

# === 输出路径（与原 ZIP 同目录）===
output_path = zip_path.with_name("knowledge_nonascii.tsv")

def extract_non_ascii_lines(zip_path: Path, output_path: Path):
    with zipfile.ZipFile(zip_path, "r") as zf:
        members = [m for m in zf.namelist() if m.lower().endswith(".tsv")]
        if not members:
            print(f"❌ 未在 {zip_path.name} 中找到 TSV 文件。")
            return
        member = members[0]
        print(f"📂 读取 {member} ...")

        data = zf.read(member)
        lines = data.splitlines()
        print(f"共有 {len(lines)} 行，开始筛选含非 ASCII 的行...")

        bad_lines = []
        for i, line in enumerate(lines, 1):
            if any(b > 127 for b in line):
                try:
                    s = line.decode("utf-8")
                except UnicodeDecodeError:
                    s = line.decode("latin-1", errors="replace")
                bad_lines.append(f"{i}\t{s}")

        if not bad_lines:
            print("🎉 未检测到任何非 ASCII 行。")
            return

        output_path.write_text("\n".join(bad_lines), encoding="utf-8")
        print(f"✅ 已输出 {len(bad_lines)} 行到：{output_path}")

# === 运行 ===
extract_non_ascii_lines(zip_path, output_path)


📂 读取 knowledge_graph.tsv ...
共有 44783097 行，开始筛选含非 ASCII 的行...
🎉 未检测到任何非 ASCII 行。


In [2]:
import zipfile
import pandas as pd
from io import BytesIO

# === 修改为你的 zip 文件路径 ===
zip_path = "/Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250831-8cdd6f/knowledge_graph.tsv.zip"

# === 打开 ZIP 并读取 TSV ===
with zipfile.ZipFile(zip_path, "r") as zf:
    # 找出第一个 .tsv 文件
    tsv_files = [name for name in zf.namelist() if name.endswith(".tsv")]
    if not tsv_files:
        raise FileNotFoundError("❌ ZIP 文件中未找到 .tsv 文件。")
    tsv_name = tsv_files[0]
    print(f"📄 读取 {tsv_name} ...")
    
    # 尝试 UTF-8 读取，如失败可换成 'latin-1'
    data = zf.read(tsv_name)
    df = pd.read_csv(BytesIO(data), sep="\t", dtype=str, encoding="utf-8", nrows=5)

print("=== 列名 ===")
print(list(df.columns))
print("\n=== 前几行数据（前 5 行） ===")
display(df.head())


📄 读取 knowledge_graph.tsv ...
=== 列名 ===
['raw_source_id', 'raw_target_id', 'raw_source_type', 'raw_target_type', 'relation_type', 'resource', 'pmids', 'key_sentence', 'source_id', 'source_type', 'target_id', 'target_type', 'formatted_relation_type']

=== 前几行数据（前 5 行） ===


,raw_source_id,raw_target_id,raw_source_type,raw_target_type,relation_type,resource,pmids,key_sentence,source_id,source_type,target_id,target_type,formatted_relation_type
0,SYMBOL:CCR2,ICD-11:1C80,Gene,Disease,DRUGBANK::treats::Compound:Disease,TTD,NaN,NaN,ENTREZ:729230,Gene,MONDO:0957421,Disease,DRUGBANK::treats::Compound:Disease
1,SYMBOL:F9,ICD-11:1C80,Gene,Disease,DRUGBANK::treats::Compound:Disease,TTD,NaN,NaN,ENTREZ:2158,Gene,MONDO:0957421,Disease,DRUGBANK::treats::Compound:Disease
2,SYMBOL:HPD,ICD-11:1C80,Gene,Disease,DRUGBANK::treats::Compound:Disease,TTD,NaN,NaN,ENTREZ:3242,Gene,MONDO:0957421,Disease,DRUGBANK::treats::Compound:Disease
3,SYMBOL:KRT6A,ICD-11:1C80,Gene,Disease,DRUGBANK::treats::Compound:Disease,TTD,NaN,NaN,ENTREZ:3853,Gene,MONDO:0957421,Disease,DRUGBANK::treats::Compound:Disease
4,SYMBOL:APOH,ICD-11:4A45,Gene,Disease,DRUGBANK::treats::Compound:Disease,TTD,NaN,NaN,ENTREZ:350,Gene,MONDO:0007140,Disease,DRUGBANK::treats::Compound:Disease


In [ ]:
from pathlib import Path

# === 你的 entities.tsv 文件路径 ===
entity_file = Path("/Users/zhuzhixing/KG/biomedgps-data/graph_data/entities.tsv")

def find_non_ascii_lines(tsv_path, max_preview=3):
    bad_lines = []
    total = 0
    with open(tsv_path, "rb") as f:
        for i, line in enumerate(f, 1):
            total += 1
            if any(b > 127 for b in line):  # 检测字节值大于127
                if len(bad_lines) < max_preview:
                    try:
                        s = line[:200].decode("utf-8")
                    except UnicodeDecodeError:
                        s = line[:200].decode("latin-1", errors="replace")
                    safe = ''.join(
                        ch if (ord(ch) < 128 and ch not in '\r\n\t')
                        else '?' for ch in s
                    )
                    bad_lines.append((i, safe))
    if not bad_lines:
        print(f"✅ {tsv_path.name}: 全部 {total} 行均为 ASCII。")
    else:
        print(f"❌ {tsv_path.name}: 含非 ASCII 字符，共检测到前 {len(bad_lines)} 行示例：")
        for ln, txt in bad_lines:
            print(f"   行 {ln}: {txt[:120]}")
    return bad_lines

# === 运行检查 ===
find_non_ascii_lines(entity_file)


In [4]:
import zipfile
import pandas as pd
from pathlib import Path

# === ZIP 文件路径 ===
zip_path = Path("/Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250831-8cdd6f/knowledge_graph.tsv.zip")

# === 解压 ZIP 文件 ===
with zipfile.ZipFile(zip_path, "r") as zf:
    # 获取压缩包内的 TSV 文件名
    tsv_files = [f for f in zf.namelist() if f.lower().endswith(".tsv")]
    if not tsv_files:
        raise FileNotFoundError("❌ 未在 ZIP 文件中找到 .tsv 文件")
    
    tsv_name = tsv_files[0]
    print(f"📦 找到 TSV 文件: {tsv_name}")

    # 解压到同级目录
    extract_path = zip_path.parent / Path(tsv_name).name
    zf.extract(tsv_name, path=zip_path.parent)
    print(f"✅ 已解压到: {extract_path}")

#=== 读取 TSV 文件到 DataFrame ===
try:
    knowledge_graph_ASCII = pd.read_csv(extract_path, sep="\t", dtype=str, low_memory=False, encoding="utf-8")
except UnicodeDecodeError:
    knowledge_graph_ASCII = pd.read_csv(extract_path, sep="\t", dtype=str, low_memory=False, encoding="latin-1")

print(f"✅ 成功读取 DataFrame：共 {len(knowledge_graph_ASCII):,} 行 × {len(knowledge_graph_ASCII.columns)} 列")
print("📁 DataFrame 名称：knowledge_graph_ASCII")


📦 找到 TSV 文件: knowledge_graph.tsv
✅ 已解压到: /Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250831-8cdd6f/knowledge_graph.tsv
✅ 成功读取 DataFrame：共 44,783,096 行 × 13 列
📁 DataFrame 名称：knowledge_graph_ASCII


In [12]:
temp_dir = "/Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250831-8cdd6f/"
temp_dir

'/Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250831-8cdd6f/'

In [13]:
import os
import pandas as pd
import unicodedata
from pathlib import Path

# --- inputs ---
# Read the kg file
kg_file_ascii = knowledge_graph_ASCII                    # your DataFrame (already loaded)
output_dir = Path(temp_dir) if "temp_dir" in globals() else Path.cwd()

# --- output paths ---
kg_file_ascii_path   = output_dir / "knowledge_graph_ascii.tsv"      # rows with ASCII-only key_sentence
kg_file_noascii_path = output_dir / "knowledge_graph_noascii.tsv"    # rows with non-ASCII key_sentence
kg_file_clean_path   = output_dir / "knowledge_graph_clean.tsv"      # final merged (order preserved)

# --- helpers ---
def translit_to_ascii(s: str):
    """Transliterate to ASCII: é→e, β→b, μ→u, quotes to plain, etc."""
    if pd.isna(s):
        return s
    s = unicodedata.normalize("NFKD", str(s))
    return "".join(ch for ch in s if ord(ch) < 128)

# 1) build a single mask (vectorized)
ks = kg_file_ascii["key_sentence"].astype(str)
mask_nonascii = ks.str.contains(r"[^\x00-\x7F]", regex=True, na=False)
n_bad = int(mask_nonascii.sum())
print(f"Rows with non-ASCII in key_sentence: {n_bad}")


Rows with non-ASCII in key_sentence: 1402


In [14]:
# 2) (optional) write split views without changing order
#    Comment these two lines out if you don't want the split files.
kg_file_ascii.loc[~mask_nonascii].to_csv(kg_file_ascii_path, sep="\t", index=False, encoding="utf-8")
kg_file_ascii.loc[mask_nonascii].to_csv(kg_file_noascii_path, sep="\t", index=False, encoding="utf-8")

# 3) transform only the affected cells in-place (no concat, no sort)
df_clean = kg_file_ascii.copy()
if n_bad:
    ks_clean = ks.copy()
    ks_clean.loc[mask_nonascii] = ks_clean.loc[mask_nonascii].map(translit_to_ascii)
    df_clean["key_sentence"] = ks_clean

# 4) write final clean file (order preserved)
df_clean.to_csv(kg_file_clean_path, sep="\t", index=False, encoding="utf-8")
print(f"✅ Written: {kg_file_clean_path}")

# 5) quick verification on key_sentence
remaining = df_clean["key_sentence"].astype(str).str.contains(r"[^\x00-\x7F]", regex=True, na=False).sum()
print(f"Remaining rows with non-ASCII in key_sentence: {remaining}")

# (optional) make downstream variable point to the cleaned file
kg_file = str(kg_file_clean_path)

✅ Written: /Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250831-8cdd6f/knowledge_graph_clean.tsv
Remaining rows with non-ASCII in key_sentence: 0


In [ ]:
import os
import pandas as pd
import numpy as np
import unicodedata

# --- paths (adjust temp_dir if needed) ---
kg_file_ascii   = os.path.join(temp_dir, "knowledge_graph_ascii.tsv")
kg_file_noascii = os.path.join(temp_dir, "knowledge_graph_noascii.tsv")
kg_file_clean   = os.path.join(temp_dir, "knowledge_graph_clean.tsv")

def translit_to_ascii(s):
    """Transliterate to ASCII (é→e, β→b, μ→u, …)."""
    if pd.isna(s):
        return s
    s = unicodedata.normalize("NFKD", str(s))
    return "".join(ch for ch in s if ord(ch) < 128)

# 1) Build a single boolean mask (vectorized; very fast)
ks = knowledge_graph_ASCII["key_sentence"].astype(str)
mask_nonascii = ks.str.contains(r"[^\x00-\x7F]", regex=True, na=False)
n_bad = int(mask_nonascii.sum())
print(f"Rows with non-ASCII in key_sentence: {n_bad}")

# 2) (Optional) write split views without copying the whole frame
#    These lines create no giant intermediate DataFrame; they stream to disk.
knowledge_graph_ASCII.loc[~mask_nonascii].to_csv(
    kg_file_ascii, sep="\t", index=False, encoding="utf-8"
)
knowledge_graph_ASCII.loc[mask_nonascii].to_csv(
    kg_file_noascii, sep="\t", index=False, encoding="utf-8"
)

# 3) Clean only the affected cells (no full-frame apply/map)
#    We modify a single Series and assign it back — minimal extra memory.
ks_clean = ks.copy()
if n_bad:
    ks_clean.loc[mask_nonascii] = ks_clean.loc[mask_nonascii].map(translit_to_ascii)

# 4) Assign back (order preserved), then write the final clean file
knowledge_graph_clean = knowledge_graph_ASCII.copy()
knowledge_graph_clean["key_sentence"] = ks_clean
knowledge_graph_clean.to_csv(kg_file_clean, sep="\t", index=False, encoding="utf-8")

# 5) Verify (vectorized)
remaining = knowledge_graph_clean["key_sentence"].astype(str)\
    .str.contains(r"[^\x00-\x7F]", regex=True, na=False).sum()
print(f"Remaining rows with non-ASCII in key_sentence: {remaining}")
print(f"✅ Written:\n  {kg_file_ascii}\n  {kg_file_noascii}\n  {kg_file_clean}")

# (Optional) enforce pure ASCII on disk — switch to encoding="ascii" on the final write above.
